# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadFaizan0023/FlyRank_ML_internship_repo/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: 4 and 5 (Logistic Regression and Decision Trees)
Reason: Handling Multi-class classification task lane as my final output - 'decline_Score' range from (0-5)

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, roc_auc_score

In [3]:
import pandas as pd
import os, getpass
import duckdb

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Time-aware split: latest 90 days window with 60 days training and 30 days testing rows. The training data will not have any client_id or content_id for training but will be included in testing data for testing and evaluation.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
data = pd.read_csv("baseline_action_score_f.csv")

/tmp/ipykernel_1092/3274622562.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("baseline_action_score_f.csv")


In [8]:
data.head(10)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ctr,engagement_rate,scroll_rate,days_since_update,position_bucket,engagement_bucket,decline_score
0,2026-05-11,client_73cda7b4e4f265ea,content_c036f9d8a7bc0396,18,13.222222,0.0,0.0,0.0,131,11-20,NaN,5
1,2026-04-01,client_9958f0a7ae1df715,content_d06983da58555f50,8,10.000000,0.0,0.0,0.0,38,4-10,NaN,5
2,2026-04-01,client_9958f0a7ae1df715,content_e9d1789ebf0ff1f6,1,10.000000,0.0,0.0,0.0,38,4-10,NaN,5
3,2026-04-01,client_9958f0a7ae1df715,content_70f2cea7bfc5dfb6,4,68.250000,0.0,0.0,0.0,49,21-100,NaN,5
4,2026-04-01,client_9958f0a7ae1df715,content_cd046a34819b45af,5,41.200000,0.0,0.0,0.0,49,21-100,NaN,5
...,...,...,...,...,...,...,...,...,...,...,...,...
995,2026-05-11,client_23a62021009f63c4,content_172f37f1364b3e36,61,18.114754,0.0,0.0,0.0,131,11-20,NaN,5
996,2026-05-11,client_23a62021009f63c4,content_b1987150086cd391,20,34.250000,0.0,0.0,0.0,131,21-100,NaN,5
997,2026-05-11,client_23a62021009f63c4,content_c0949943ca59081f,11,14.909091,0.0,0.0,0.0,131,11-20,NaN,5
998,2026-04-01,client_20259bd6705d81d4,content_7fb357fe837215bf,23,41.956522,0.0,0.0,0.0,47,21-100,NaN,5


In [6]:
data.tail(10)

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ctr,engagement_rate,scroll_rate,days_since_update,position_bucket,engagement_bucket,decline_score
1683287,2026-05-30,client_1a730cb2640a1abf,content_63bc11c45c14e8fe,1720,7.182558,0.000581,0.250000,0.250000,26,4-10,low,0
1683288,2026-06-28,client_73cda7b4e4f265ea,content_c2799e6502d2de3f,132,5.537879,0.007576,1.000000,1.000000,1,4-10,very_high,0
1683289,2026-04-24,client_23a62021009f63c4,content_138abe69ee5bfd14,924,7.462121,0.009740,0.083333,0.040000,14,4-10,low,0
1683290,2026-04-01,client_3f0ce4d44fe94f3d,content_b2347ac65344f8c0,446,1.914798,0.002242,1.000000,1.000000,12,1-3,very_high,0
1683291,2026-05-30,client_1a730cb2640a1abf,content_f233334aec6c9b56,2438,2.150123,0.001641,0.111111,0.111111,26,1-3,low,0
1683292,2026-05-30,client_1a730cb2640a1abf,content_54c0fa31d867a8ea,281,4.209964,0.010676,0.200000,0.200000,26,4-10,low,0
1683293,2026-05-30,client_1a730cb2640a1abf,content_d9ff67b581e72d74,176,7.573864,0.011364,0.500000,0.500000,26,4-10,medium,0
1683294,2026-04-24,client_23a62021009f63c4,content_dd7c5f83b5b50ee6,115,8.878261,0.026087,0.500000,0.250000,14,4-10,medium,0
1683295,2026-05-15,client_73cda7b4e4f265ea,content_2fe75239353ca721,710,3.773239,0.005634,0.166667,0.375000,7,4-10,low,0
1683296,2026-06-25,client_73cda7b4e4f265ea,content_381e41a7ae8bfb12,345,5.666667,0.002899,1.000000,1.000000,8,4-10,very_high,0


In [9]:
data.columns

Index(['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions',
       'gsc_avg_position', 'ctr', 'engagement_rate', 'scroll_rate',
       'days_since_update', 'position_bucket', 'engagement_bucket',
       'decline_score'],
      dtype='object')

In [11]:
data['report_date'] = pd.to_datetime(data['report_date'])

train_data = data[(data['report_date'] >= '2026-04-01') & (data['report_date'] <= '2026-05-30')].copy()

test_data = data[(data['report_date'] >= '2026-06-01') & (data['report_date'] <= '2026-06-30')].copy()

print(f"Training set size: {len(train_data)}")
print(f"Testing set size: {len(test_data)}")
display(train_data.head())
display(test_data.head())

Training set size: 1111616
Testing set size: 554216


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ctr,engagement_rate,scroll_rate,days_since_update,position_bucket,engagement_bucket,decline_score
0,2026-05-11,client_73cda7b4e4f265ea,content_c036f9d8a7bc0396,18,13.222222,0.0,0.0,0.0,131,11-20,NaN,5
1,2026-04-01,client_9958f0a7ae1df715,content_d06983da58555f50,8,10.000000,0.0,0.0,0.0,38,4-10,NaN,5
2,2026-04-01,client_9958f0a7ae1df715,content_e9d1789ebf0ff1f6,1,10.000000,0.0,0.0,0.0,38,4-10,NaN,5
3,2026-04-01,client_9958f0a7ae1df715,content_70f2cea7bfc5dfb6,4,68.250000,0.0,0.0,0.0,49,21-100,NaN,5
4,2026-04-01,client_9958f0a7ae1df715,content_cd046a34819b45af,5,41.200000,0.0,0.0,0.0,49,21-100,NaN,5


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_avg_position,ctr,engagement_rate,scroll_rate,days_since_update,position_bucket,engagement_bucket,decline_score
76,2026-06-30,client_a22068e339bf95f5,content_4226eaa46864a67f,15,19.333333,0.0,0.0,0.0,47,11-20,NaN,5
132,2026-06-30,client_7de9989c909e91a5,content_61596e2dda4e4008,8,14.125000,0.0,0.0,0.0,35,11-20,NaN,5
133,2026-06-30,client_7de9989c909e91a5,content_ddd4062b6401587c,12,21.500000,0.0,0.0,0.0,35,21-100,NaN,5
134,2026-06-30,client_7de9989c909e91a5,content_6d9e6b27becbe515,16,18.562500,0.0,0.0,0.0,35,11-20,NaN,5
135,2026-06-30,client_7de9989c909e91a5,content_a4ed21325c168ae7,1,10.000000,0.0,0.0,0.0,35,4-10,NaN,5


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
X_train = train_data.drop(columns=['report_date', 'client_hash_id', 'content_hash_id', 'decline_score'], axis=1)
y_train = train_data['decline_score']

In [24]:
print(len(X_train))
print(len(y_train))

1111616
1111616


In [28]:
X_test = test_data.drop(columns=['report_date', 'client_hash_id', 'content_hash_id', 'decline_score'], axis=1)

In [29]:
print(len(X_test))

554216


In [ ]:
model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)               # binary predictions -> for F1
y_proba = model.predict_proba(X_test)[:, 1]  # positive-class probability -> for ROC-AUC

f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f"F1: {f1:.4f}")
print(f"ROC-AUC: {auc:.4f}")

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.